# Pertemuan 1 — Pendahuluan dan Notasi Asimtotik
**Capaian:** menghitung kompleksitas potongan kode, mengukur waktu eksekusi empiris, dan menghubungkannya dengan Big-O / Θ / Ω.

**Alur praktikum:** (1) hitung operasi → (2) ukur waktu → (3) best/avg/worst case → (4) verifikasi Θ numerik → (5) biaya komputasi AI → (6) latihan.

In [ ]:
import time, math, random
import numpy as np
import matplotlib.pyplot as plt
random.seed(42)

## 1. Menghitung operasi dasar (model RAM)
Alih-alih waktu (bergantung mesin), kita **hitung operasi** dengan penghitung.

In [ ]:
def lin(n):
    ops = 0
    for i in range(n): ops += 1
    return ops
def quad(n):
    ops = 0
    for i in range(n):
        for j in range(n): ops += 1
    return ops
def logn(n):
    ops, i = 0, 1
    while i < n: ops += 1; i *= 2
    return ops
def nlogn(n):
    ops = 0
    for i in range(n):
        j = 1
        while j < n: ops += 1; j *= 2
    return ops

print(f"{'n':>6} {'O(log n)':>9} {'O(n)':>8} {'O(n log n)':>11} {'O(n^2)':>10}")
for n in [10, 100, 500, 1000]:
    print(f"{n:>6} {logn(n):>9} {lin(n):>8} {nlogn(n):>11} {quad(n):>10}")

## 2. Pengukuran waktu empiris + estimasi pangkat
Pada skala log-log, fungsi `c·nᵏ` berupa garis lurus dengan **kemiringan k**.

In [ ]:
def measure(fn, n, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        t = time.perf_counter(); fn(n); best = min(best, time.perf_counter() - t)
    return best

ns = [200, 400, 800, 1600]
t_lin  = [measure(lin,  n * 200) for n in ns]   # n diperbesar agar terukur
t_quad = [measure(quad, n)       for n in ns]
for name, ts, base in [("linear", t_lin, [n * 200 for n in ns]), ("kuadratik", t_quad, ns)]:
    k = np.polyfit(np.log(base), np.log(ts), 1)[0]
    print(f"{name:10s} perkiraan pangkat k = {k:.2f}")
plt.loglog(ns, t_lin, "o-", label="O(n)"); plt.loglog(ns, t_quad, "s-", label="O(n²)")
plt.xlabel("n"); plt.ylabel("detik"); plt.legend(); plt.grid(True, which="both"); plt.show()

## 3. Best / average / worst case — Linear Search

In [ ]:
def linear_search(a, x):
    cmp = 0
    for v in a:
        cmp += 1
        if v == x: return cmp
    return cmp

n = 1000
a = list(range(n))
best, worst = linear_search(a, a[0]), linear_search(a, -1)
avg = np.mean([linear_search(a, random.choice(a)) for _ in range(5000)])
print("best =", best, "| worst =", worst, "| average ≈", round(avg, 1), "≈ (n+1)/2 =", (n + 1) / 2)

## 4. Verifikasi numerik f(n) = Θ(g(n))
Cari konstanta c₁, c₂, n₀ sehingga c₁·g(n) ≤ f(n) ≤ c₂·g(n) untuk n ≥ n₀.

In [ ]:
f = lambda n: 3 * n**2 + 10 * n + 5
g = lambda n: n**2
ratios = [f(n) / g(n) for n in range(1, 10001)]
print("rasio f/g untuk n ≥ 1  → min =", round(min(ratios), 3), " max =", round(max(ratios), 3))
print("Ambil c1 = 3, c2 = 18, n0 = 1  ⇒ f(n) = Θ(n²)")
n_cross = next(n for n in range(1, 10**4) if 100 * n < n**2)
print("100n < n² mulai dari n =", n_cross, "→ konstanta besar bisa menang untuk n kecil")

## 5. [Modern] Mengapa kompleksitas penting di era AI
Pendekatan umum (Kaplan dkk., 2020): FLOPs *training* Transformer ≈ **6·N·D** (N = parameter, D = token); *inference* ≈ 2·N FLOPs per token. Angka di bawah hanyalah **perkiraan orde besar**.

In [ ]:
def train_flops(N, D): return 6 * N * D
for N, D in [(1e8, 2e9), (7e9, 1e12), (7e10, 2e12)]:
    fl = train_flops(N, D)
    gpu_days = fl / (1e14 * 0.4) / 86400     # asumsi 100 TFLOPs puncak, utilisasi 40%
    print(f"N={N:.0e}, D={D:.0e} → {fl:.2e} FLOPs ≈ {gpu_days:,.1f} GPU-hari (asumsi 100 TFLOPs @40%)")

## 6. Latihan
1. Tentukan kompleksitas (hitung operasi lalu verifikasi dengan `measure`) untuk tiga fungsi di bawah.
2. Buat fungsi `fit_exponent(fn, ns)` yang mengembalikan perkiraan k.
3. Tunjukkan dengan eksperimen bahwa `x in list` O(n) sedangkan `x in set` O(1) rata-rata.

In [ ]:
def soal_a(n):
    s = 0
    for i in range(n):
        for j in range(i): s += 1
    return s
def soal_b(n):
    s, i = 0, n
    while i > 0: s += i; i //= 2
    return s
def soal_c(n):
    s = 0
    for i in range(1, n):
        for j in range(0, n, i): s += 1
    return s
def fit_exponent(fn, ns):
    pass  # TODO

**Refleksi:** Mengapa pengukuran waktu saja tidak cukup untuk menyimpulkan kelas kompleksitas? Kapan algoritma O(n²) bisa lebih cepat daripada O(n log n) dalam praktik?